# Day 19: Handling RAG Context Overflow with Sliding Windows

## Core Theory (Just-in-Time)
When building Retrieval-Augmented Generation (RAG) systems, a common issue is **Context Overflow**. Large Language Models (LLMs) have a maximum context window limit (e.g., 4k, 8k, 128k tokens). If your combined retrieved documents exceed this limit, the API call will fail or the model will forget the beginning of the context (a phenomenon known as "Lost in the Middle").

To solve this natively before passing the prompt to the LLM, we can implement a **Sliding Window** approach over the retrieved text. Instead of passing all documents as one giant string, we slice the text into manageable, overlapping chunks, or simply truncate the retrieved documents list until it fits within our context budget. Understanding how to manage this in standard Python ensures you are not overly reliant on "black box" abstractions.

In [1]:
from typing import List

def truncate_context_sliding_window(retrieved_texts: List[str], max_words: int = 100) -> str:
    """
    Combines retrieved texts and truncates them to a maximum word count to prevent context overflow.
    In production, you would swap the word count for an exact token count (e.g., via tiktoken).
    
    Args:
        retrieved_texts: A list of document strings.
        max_words: Maximum allowed words for the combined context (approximation for tokens).
        
    Returns:
        A single string containing the combined and safely truncated context.
    """
    combined_words: List[str] = []
    
    for text in retrieved_texts:
        words = text.split()
        if len(combined_words) + len(words) > max_words:
            # Slice the words to fit exactly within the remaining allowance
            allowance = max_words - len(combined_words)
            combined_words.extend(words[:allowance])
            break  # Context window is full
        
        combined_words.extend(words)
        
    return " ".join(combined_words)


In [2]:
# Example Usage:
retrieved_docs = [
    "This is the first highly relevant document that we retrieved from Qdrant.",
    "This is the second document with more specific details about the architecture.",
    "This is a third document that might be too long to fit into our small simulated window."
]

# Set a deliberately small max_words to demonstrate truncation
safe_context = truncate_context_sliding_window(retrieved_docs, max_words=20)
print("--- Safe Context ---")
print(safe_context)


--- Safe Context ---
This is the first highly relevant document that we retrieved from Qdrant. This is the second document with more specific


## Common Pitfalls in Production
1. **Blind Truncation:** Simply chopping off text at an arbitrary character count often splits words or mid-sentence, leading to corrupted context and LLM hallucinations. Always use token-based or word-based truncation.
2. **Lost in the Middle:** If you pack the maximum context window full of retrieved documents, LLMs tend to ignore information in the middle. Often, it is better to strictly prioritize and send *fewer* high-quality tokens rather than filling the entire available context window.
3. **Tokenizer Mismatch:** Using an outdated or mismatched tokenizer (e.g., using a BERT tokenizer for an OpenAI model) will result in inaccurate token counts, causing context overflow errors during the actual API call. While we used words above for demonstration, production systems require model-specific tokenizers.

## Practical Lab / Homework
**Your Task:** 
Implement a function `sliding_window_chunks` that takes a long single document and returns overlapping chunks of it, ensuring no single chunk exceeds `window_size_words`, and consecutive chunks overlap by `overlap_words`.

In [3]:
from typing import List

def sliding_window_chunks(document: str, window_size_words: int = 15, overlap_words: int = 5) -> List[str]:
    """
    Splits a document into overlapping word-based chunks.
    
    Args:
        document: The large input text to chunk.
        window_size_words: The maximum number of words per chunk.
        overlap_words: The number of overlapping words between consecutive chunks.
        
    Returns:
        A list of string chunks.
    """
    words = document.split()
    chunks: List[str] = []
    start = 0
    total_words = len(words)
    
    while start < total_words:
        end = start + window_size_words
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))
        
        if end >= total_words:
            break
            
        start += (window_size_words - overlap_words)
        
    return chunks

# Lab Verification
long_document = "AI Engineering requires understanding how to effectively manage context limits in large language models properly. " * 3
chunks = sliding_window_chunks(long_document, window_size_words=10, overlap_words=3)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {repr(chunk)}")


Chunk 1: 'AI Engineering requires understanding how to effectively manage context limits'
Chunk 2: 'manage context limits in large language models properly. AI Engineering'
Chunk 3: 'properly. AI Engineering requires understanding how to effectively manage context'
Chunk 4: 'effectively manage context limits in large language models properly. AI'
Chunk 5: 'models properly. AI Engineering requires understanding how to effectively manage'
Chunk 6: 'to effectively manage context limits in large language models properly.'
